In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import cv2
BASE_PATH = "/content/drive/MyDrive/physalis_project/classification_raw/02_physalis_madurez"
OUTPUT_PATH = "/content/drive/MyDrive/physalis_project/classification_dataset"
classes = {
    0: "INTERMEDIA",
    1: "MADURO",
    2: "NO_APTA",
    3: "VERDE"
}
splits = ["train", "valid", "test"]
for split in splits:
    for cls in classes.values():
        os.makedirs(os.path.join(OUTPUT_PATH, split, cls), exist_ok=True)
def yolo_to_bbox(img_w, img_h, x, y, w, h):
    x1 = int((x - w/2) * img_w)
    y1 = int((y - h/2) * img_h)
    x2 = int((x + w/2) * img_w)
    y2 = int((y + h/2) * img_h)
    return x1, y1, x2, y2
total_crops = 0
for split in splits:
    img_dir = os.path.join(BASE_PATH, split, "images")
    lbl_dir = os.path.join(BASE_PATH, split, "labels")
    if not os.path.exists(img_dir) or not os.path.exists(lbl_dir):
        continue
    for file in os.listdir(img_dir):
        if not file.endswith((".jpg", ".png", ".jpeg")):
            continue
        img_path = os.path.join(img_dir, file)
        lbl_path = os.path.join(lbl_dir, file.replace(".jpg", ".txt"))
        if not os.path.exists(lbl_path):
            continue
        image = cv2.imread(img_path)
        h, w, _ = image.shape
        with open(lbl_path, "r") as f:
            lines = f.readlines()
        for i, line in enumerate(lines):
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls_id, x, y, bw, bh = map(float, parts)
            cls_id = int(cls_id)
            if cls_id not in classes:
                continue
            x1, y1, x2, y2 = yolo_to_bbox(w, h, x, y, bw, bh)
            crop = image[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            save_path = os.path.join(
                OUTPUT_PATH,
                split,
                classes[cls_id],
                f"{file[:-4]}_{i}.jpg"
            )
            cv2.imwrite(save_path, crop)
            total_crops += 1
print(f"Cropping Done! Total crops: {total_crops}")

Cropping Done! Total crops: 1806


In [ ]:
!ls /content/drive/MyDrive/physalis_project/classification_dataset/train

INTERMEDIA  MADURO  NO_APTA  VERDE


In [ ]:
import os

base = "/content/drive/MyDrive/physalis_project/classification_dataset/train"

for cls in os.listdir(base):
    path = os.path.join(base, cls)
    print(cls, ":", len(os.listdir(path)))

INTERMEDIA : 295
MADURO : 303
NO_APTA : 210
VERDE : 631
